# Anthropic API Examples

Companion notebook for [`README.md`](./README.md).  
All examples use the `anthropic` library and load credentials from a `.env` file.

**Requirements:** `conda activate agents` and `pip install anthropic python-dotenv`

In [1]:
# Shared setup — run this cell first
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()          # reads ANTHROPIC_API_KEY from .env
client = Anthropic()   # picks up the key automatically
print("Client ready.")

Client ready.


---
## 1. Basic Message

Every request requires `model`, `max_tokens`, and `messages`.  
The `system` prompt is a **top-level parameter** — not a message role.

In [2]:
message = client.messages.create(
    model="claude-haiku-4-5",
    max_tokens=256,
    system="You are a concise assistant.",
    messages=[
        {"role": "user", "content": "What is the capital of Japan?"},
    ],
)

print(message.content[0].text)
print(f"\nStop reason:    {message.stop_reason}")
print(f"Input tokens:   {message.usage.input_tokens}")
print(f"Output tokens:  {message.usage.output_tokens}")

Tokyo is the capital of Japan.

Stop reason:    end_turn
Input tokens:   21
Output tokens:  10


---
## 2. Multi-Turn Conversation

Maintain context by appending each assistant reply to `messages`.  
Roles must alternate: `user` → `assistant` → `user` → …

In [3]:
messages = []

def chat(user_input: str) -> str:
    messages.append({"role": "user", "content": user_input})
    response = client.messages.create(
        model="claude-haiku-4-5",
        max_tokens=256,
        system="You are a helpful assistant.",
        messages=messages,
    )
    reply = response.content[0].text
    messages.append({"role": "assistant", "content": reply})
    return reply

print(chat("My name is Alice."))
print()
print(chat("What is my name?"))   # model should remember

Nice to meet you, Alice! How can I help you today?

Your name is Alice, as you told me at the beginning of our conversation.


---
## 3. Streaming

### 3a. `.stream()` context manager

Recommended approach — provides a `text_stream` iterator and access to the final message with usage stats.

In [4]:
with client.messages.stream(
    model="claude-haiku-4-5",
    max_tokens=256,
    messages=[{"role": "user", "content": "Write a haiku about Python."}],
) as stream:
    for text in stream.text_stream:
        print(text, end="", flush=True)
print()

final = stream.get_final_message()
print(f"\nInput tokens:  {final.usage.input_tokens}")
print(f"Output tokens: {final.usage.output_tokens}")

# Python

Serpent of code flows,
Elegant syntax winds through—
Logic made simple.

Input tokens:  14
Output tokens: 25


### 3b. Low-level `stream=True`

Iterates raw server-sent events. Useful when you need full control over event types.

In [5]:
with client.messages.create(
    model="claude-haiku-4-5",
    max_tokens=256,
    messages=[{"role": "user", "content": "List three benefits of Python."}],
    stream=True,
) as stream:
    for event in stream:
        if event.type == "content_block_delta":
            print(event.delta.text, end="", flush=True)
print()

# Three Benefits of Python

1. **Easy to Learn and Read**
   - Python's simple, clean syntax resembles natural English, making it ideal for beginners and reducing the learning curve compared to other programming languages.

2. **Versatile and Widely Used**
   - Python supports multiple programming paradigms and is used across many fields including web development, data science, artificial intelligence, automation, and scientific computing.

3. **Large Community and Rich Ecosystem**
   - Python has an extensive library of ready-made packages (like NumPy, Django, and Pandas) and a large, active community that provides abundant resources, tutorials, and support.


---
## 4. Tool Use / Function Calling

1. Define tools with `input_schema`.  
2. First call: model returns `stop_reason = "tool_use"`.  
3. Execute the function locally.  
4. Return a `tool_result` message to get the final answer.

In [6]:
import json

tools = [
    {
        "name": "get_weather",
        "description": "Return current weather for a city.",
        "input_schema": {
            "type": "object",
            "properties": {
                "city": {"type": "string", "description": "City name"},
                "unit": {"type": "string", "enum": ["celsius", "fahrenheit"]},
            },
            "required": ["city"],
        },
    }
]

messages = [{"role": "user", "content": "What's the weather in Tokyo?"}]

# First call
response = client.messages.create(
    model="claude-haiku-4-5",
    max_tokens=256,
    tools=tools,
    messages=messages,
)
print("Stop reason:", response.stop_reason)

if response.stop_reason == "tool_use":
    tool_block = next(b for b in response.content if b.type == "tool_use")
    print(f"Tool called: {tool_block.name} | {tool_block.input}")

    # Simulate the actual function
    tool_result = {"temperature": 18, "condition": "Cloudy", "city": tool_block.input["city"]}

    messages.append({"role": "assistant", "content": response.content})
    messages.append({
        "role": "user",
        "content": [{
            "type": "tool_result",
            "tool_use_id": tool_block.id,
            "content": json.dumps(tool_result),
        }],
    })

    # Second call — final answer
    final = client.messages.create(
        model="claude-haiku-4-5",
        max_tokens=256,
        tools=tools,
        messages=messages,
    )
    print("\nFinal answer:", final.content[0].text)

Stop reason: tool_use
Tool called: get_weather | {'city': 'Tokyo'}

Final answer: The weather in Tokyo is currently:
- **Temperature:** 18°C
- **Condition:** Cloudy

It's a mild, overcast day in Tokyo.


---
## 5. Structured Output with Pydantic

Use `client.messages.parse()` with `output_format` set to a Pydantic model.  
The SDK handles JSON schema generation and parsing automatically.

In [7]:
import pydantic

class CalendarEvent(pydantic.BaseModel):
    title: str
    date: str
    time: str
    attendees: list[str]
    location: str | None = None

parsed = client.messages.parse(
    model="claude-sonnet-4-5",
    max_tokens=512,
    output_format=CalendarEvent,
    messages=[{
        "role": "user",
        "content": "Extract: Team meeting tomorrow at 3pm with Alice and Bob in Room A.",
    }],
)

event = parsed.parsed_output
print(f"Title:     {event.title}")
print(f"Date:      {event.date}")
print(f"Time:      {event.time}")
print(f"Attendees: {', '.join(event.attendees)}")
print(f"Location:  {event.location}")

Title:     Team meeting
Date:      tomorrow
Time:      3pm
Attendees: Alice, Bob
Location:  Room A


---
## 6. Vision (Image Input)

Pass images in `content` alongside text. Supports public URLs and base64 local files.

### 6a. Image from URL

In [8]:
response = client.messages.create(
    model="claude-haiku-4-5",
    max_tokens=256,
    messages=[{
        "role": "user",
        "content": [
            {
                "type": "image",
                "source": {
                    "type": "url",
                    "url": "https://upload.wikimedia.org/wikipedia/commons/thumb/d/d5/2023_06_08_Raccoon1.jpg/400px-2023_06_08_Raccoon1.jpg",
                },
            },
            {"type": "text", "text": "What animal is in this image?"},
        ],
    }],
)
print(response.content[0].text)

This is a **raccoon**. You can identify it by its distinctive features, including the characteristic black "mask" markings across its eyes and its grayish-brown fur. In this image, the raccoon appears to be on or near a tree trunk, which is typical behavior as raccoons are excellent climbers and often forage in trees at night.


### 6b. Image from local file (base64)

In [9]:
import base64

# Replace with a real local image path to test
image_path = "../openai_api/assets/2023_06_08_Raccoon1.jpg"

try:
    with open(image_path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode()

    response = client.messages.create(
        model="claude-haiku-4-5",
        max_tokens=256,
        messages=[{
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "source": {
                        "type": "base64",
                        "media_type": "image/jpeg",
                        "data": b64,
                    },
                },
                {"type": "text", "text": "Describe this image."},
            ],
        }],
    )
    print(response.content[0].text)
except FileNotFoundError:
    print(f"File not found: {image_path} — replace with a real image path to test.")

# Image Description

This photograph captures a **raccoon** peering out from behind the textured bark of a large tree trunk. The image features:

**Composition:**
- The raccoon is positioned against the left side of the frame, emerging from behind the tree
- Dark background creates strong contrast, making the subject stand out
- The tree's rough, gray bark texture dominates the left portion of the image

**The Raccoon:**
- Clearly visible distinctive features: the characteristic black "mask" marking across its eyes
- Rounded ears and alert facial expression
- The animal appears to be observing its surroundings cautiously
- Typical nocturnal behavior—likely photographed at dusk or with specialized lighting

**Lighting:**
- Professional wildlife photography with deliberate lighting that illuminates the raccoon while keeping the background dark
- Creates a dramatic, naturalistic effect

This is a well-composed wildlife photograph that captures the raccoon's curious and somewhat mischievou

---
## 7. Extended Thinking

Enable Claude to reason internally before answering by setting `thinking.type = "enabled"` and allocating a `budget_tokens` for internal reasoning.  
The response includes a `thinking` block followed by the final `text` block.

> Requires `claude-sonnet-4-5` or later. `max_tokens` must exceed `budget_tokens`.

In [10]:
response = client.messages.create(
    model="claude-sonnet-4-5",
    max_tokens=8000,
    thinking={
        "type": "enabled",
        "budget_tokens": 5000,   # tokens reserved for internal reasoning
    },
    messages=[{
        "role": "user",
        "content": "A train leaves Chicago at 9am at 60 mph. Another leaves New York at 10am at 80 mph toward Chicago (800 miles apart). When and where do they meet?",
    }],
)

for block in response.content:
    if block.type == "thinking":
        print("=== Thinking (first 400 chars) ===")
        print(block.thinking[:400], "...\n")
    elif block.type == "text":
        print("=== Answer ===")
        print(block.text)

=== Thinking (first 400 chars) ===
Let me set up this problem carefully.

- Train 1 leaves Chicago at 9am, traveling at 60 mph toward New York
- Train 2 leaves New York at 10am, traveling at 80 mph toward Chicago
- The cities are 800 miles apart

Let me set Chicago as position 0 and New York as position 800.

Train 1 travels from Chicago toward New York (increasing position)
Train 2 travels from New York toward Chicago (decreasing  ...

=== Answer ===
Looking at this problem, I need to find when the trains meet.

**Setting up the problem:**
- Train 1 (Chicago): starts at 9am, speed = 60 mph
- Train 2 (New York): starts at 10am, speed = 80 mph  
- Distance between cities = 800 miles

**Let t = hours after 9am**

Position of Train 1 from Chicago: 60t

Position of Train 2 from Chicago: 800 - 80(t-1)
- Note: (t-1) because it starts 1 hour later

**When they meet:**
60t = 800 - 80(t-1)
60t = 800 - 80t + 80
140t = 880
t = 880/140 = 44/7 ≈ 6.29 hours

**Time:** 9am + 6.29 hours = **3:17pm** (

---
## 8. Async Client

Use `AsyncAnthropic` for non-blocking calls in async frameworks (FastAPI, asyncio).  

> In Jupyter, `await` works directly in cells — no `asyncio.run()` needed.

In [11]:
from anthropic import AsyncAnthropic

async_client = AsyncAnthropic()   # also reads ANTHROPIC_API_KEY from env

# Basic async message
response = await async_client.messages.create(
    model="claude-haiku-4-5",
    max_tokens=256,
    messages=[{"role": "user", "content": "Name three planets."}],
)
print(response.content[0].text)

Here are three planets:

1. **Earth**
2. **Mars**
3. **Jupiter**


In [12]:
# Async streaming
async with async_client.messages.stream(
    model="claude-haiku-4-5",
    max_tokens=256,
    messages=[{"role": "user", "content": "Count to five, one word per line."}],
) as stream:
    async for text in stream.text_stream:
        print(text, end="", flush=True)
print()

One
Two
Three
Four
Five


---
## 9. Token Counting

Estimate token usage **before** making a call — useful for context limit checks and cost estimation.

In [13]:
count = client.messages.count_tokens(
    model="claude-haiku-4-5",
    system="You are a helpful assistant.",
    messages=[
        {"role": "user",      "content": "Hello, how are you?"},
        {"role": "assistant", "content": "I'm doing well, thank you!"},
        {"role": "user",      "content": "Can you explain quantum entanglement?"},
    ],
)

print(f"Estimated input tokens: {count.input_tokens}")

Estimated input tokens: 41


---
## Model Reference

| Model | Context | Best for |
|---|---|---|
| `claude-opus-4-5` | 200k | Most capable, complex tasks |
| `claude-sonnet-4-5` | 200k | Balanced performance/cost |
| `claude-haiku-4-5` | 200k | Fast, cost-efficient |

> Check [platform.claude.com/docs/en/models/overview](https://platform.claude.com/docs/en/models/overview) for the latest model list and exact version strings (e.g., `claude-haiku-4-5-20251001`).